In [ ]:
#======= InnomaticsResearchLabs_EntranceTest========#
#======= Candidate Name: Shaik Mazher       ========#

import pandas as pd
import sqlite3

#Loading Datasets
#----------------

#load order file
df_order=pd.read_csv('orders.csv')
print("order file is loaded :",df_order.shape)

#load json file
df_json=pd.read_json('users.json')
print("json file is loaded :",df_json.shape)

#load restaurant file (SQL data)
conn = sqlite3.connect(':Memory:')
conn.execute("DROP TABLE IF EXISTS restaurants")

with open("restaurants.sql",'r') as f:
  conn.executescript(f.read())

df_rest=pd.read_sql_query("SELECT * FROM restaurants",conn)
print("restaurant file loaded :",df_rest.shape)

#DATA MERGING
#------------

#marging files
df_merge=df_order.merge(df_json, on='user_id', how='left')
final_merging=df_merge.merge(df_rest, on='restaurant_id' , how='left')
print("final merging is done :",final_merging.shape)

#saving all the data in a master dataset file
final_merging.to_csv('master_dataset.csv',index=False)
print("The Dataset is loaded!")

# SOLVING MULTIPLE CHOICE QUESTIONS

# 1. Which city has the highest total revenue (total_amount) from Gold members?
gold_members=final_merging[final_merging['membership'] == 'Gold']
city_revenue=gold_members.groupby('city')['total_amount'].sum().sort_values(ascending=False)
print("All cities Revenue from gole members is:")
print(city_revenue)

# 2.Which cuisine has the highest average order value across all orders?
cuisine_avg=final_merging.groupby('cuisine')['total_amount'].mean().sort_values(ascending=False)
print("All cuisines order value is:")
print(cuisine_avg)

#3. How many distinct users placed orders worth more than ₹1000 in total (sum of all their orders)?
distinct_users=final_merging.groupby("user_id")['total_amount'].sum()
high_val_user=distinct_users[distinct_users > 1000]
print("The distinct users are:")
print(high_val_user.count())

#4. Which restaurant rating range generated the highest total revenue?
highest_rating=final_merging.groupby("rating")["total_amount"].sum().sort_values(ascending=False)
print("The highest rating range is:")
print(highest_rating)

#5. Among Gold members, which city has the highest average order value?
gold_members=final_merging[final_merging['membership'] == 'Gold']
city_avg=gold_members.groupby('city')['total_amount'].mean().sort_values(ascending=False)
print("City that the heighest avg order among gold members:")
print(city_avg)

#6. Which cuisine has the lowest number of distinct restaurants but still contributes significant revenue?
distinct_restaurant=final_merging.groupby('cuisine')['restaurant_id'].nunique()

print("The lowest number of distinct restaurants but still contributes significant revenue:")
print(distinct_restaurant)

#7. What percentage of total orders were placed by Gold members? (Rounded to nearest integer)
total_orders=len(final_merging)
gold_orders=len(final_merging[final_merging['membership'] == 'Gold'])
gold_percentage=(gold_orders/total_orders)*100
print(f"Percentage of golde orders :{gold_percentage:.2f}% ")
print(f"Rounded to nearest integer Answer:{round(gold_percentage)}%")

#8. Which restaurant has the highest average order value but less than 20 total orders?
avg_order_value=final_merging.groupby('restaurant_name_y')['total_amount'].agg(['mean','count'])
rest_lessthen_20=avg_order_value[avg_order_value["count"] <20 ].sort_values(by='mean',ascending=False)
print("The restaurant has the highest average order value but less than 20 total orders is:")
print(rest_lessthen_20)

#9. Which combination contributes the highest revenue?
comb_revenue=final_merging.groupby(['membership','cuisine'])['total_amount'].sum().sort_values(ascending=False)
print("The combination  highest revenue is:")
print(comb_revenue)

#10. During which quarter of the year is the total revenue highest?
final_merging['order_date']=pd.to_datetime(final_merging['order_date'],dayfirst=True)
final_merging['month']=final_merging['order_date'].dt.month

def get_quarter(month):
  if month <=3:
    return 'Q1 (Jan-Mar)'
  elif month <=6:
    return 'Q2 (Apr-Jun)'
  elif month <=9:
    return 'Q3 (Jul-Sep)'
  else:
    return 'Q4 (Oct-Dec)'

final_merging['quarter_manual']=final_merging['month'].apply(get_quarter)
manual_revenue=final_merging.groupby('quarter_manual')['total_amount'].sum().sort_values(ascending=False)
print("The year of the total revenue highest is:")
print(manual_revenue)

#NUMERICAL QUESTIONS
#-------------------

# 11. How many total orders were placed by users with Gold membership?
gold_members=len(final_merging[final_merging['membership'] == 'Gold'])
print("The total orders by users with Gold membership is:")
print(gold_members)

#12. What is the total revenue (rounded to nearest integer) generated from orders placed in Hyderabad city?
hyd_revenue=final_merging[final_merging['city'] == 'Hyderabad']
total_Hyd_revenue=hyd_revenue["total_amount"].sum()
print("Total revenue of Hyderabad is:",round(total_Hyd_revenue))

#13. How many distinct users placed at least one order?
distinct_users=final_merging['user_id'].nunique()
print("The distinct users placed at least one order is:",distinct_users)

#14. What is the average order value (rounded to 2 decimals) for Gold members?
gold_members=final_merging[final_merging['membership'] == 'Gold']
avg_gold_order=gold_members['total_amount'].mean()
print("Average Order Value (Gold):", round(avg_gold_order, 2))

#15.How many orders were placed for restaurants with rating ≥ 4.5?
order_placed=final_merging[final_merging['rating'] >= 4.5]
high_prder_placed=len(order_placed)
print("The orders were placed for the restaurants with rating ≥ 4.5 is:",high_prder_placed)

#16.How many orders were placed in the top revenue city among Gold members only?
city_orders=final_merging[final_merging['membership'] == 'Gold']

top_revenue_city=city_orders.groupby('city')['total_amount'].sum().idxmax()
print("Top revenue city among Gold members only is:")

#count the orders in the city
count_orders=len(city_orders[city_orders['city'] == top_revenue_city] )
print("The top city revenue among gold members",count_orders)



order file is loaded : (10000, 6)
json file is loaded : (3000, 4)
restaurant file loaded : (500, 4)
final merging is done : (10000, 12)
The Dataset is loaded!
All cities Revenue from gole members is:
city
Chennai      1080909.79
Pune         1003012.32
Bangalore     994702.59
Hyderabad     896740.19
Name: total_amount, dtype: float64
All cuisines order value is:
cuisine
Mexican    808.021344
Italian    799.448578
Indian     798.466011
Chinese    798.389020
Name: total_amount, dtype: float64
The distinct users are:
2544
The highest rating range is:
rating
4.8    657707.71
4.6    495867.97
3.2    490913.01
4.5    479047.03
4.9    467467.09
3.8    466878.69
3.1    443863.92
4.2    423185.06
4.7    416301.51
4.1    380850.85
3.7    368173.17
4.4    346276.90
3.4    339942.79
4.3    330966.42
3.5    318822.05
4.0    318261.42
3.9    299987.19
3.3    288212.80
3.6    264193.94
3.0    255018.13
5.0    159686.47
Name: total_amount, dtype: float64
City that the heighest avg order among gold mem

Feb 2026 Internship – Logic Building Task 1

In [1]:
# Innomatics Task 1
# completed by Shaik Mazher

# 1. User Login Check
username = "admin"
password = "1234"

if username == "admin" and password == "1234":
    print("Login Successful")
else:
    print("Invalid Credentials")

print("-------------------")

# 2. Pass / Fail Analyzer
marks = [45, 78, 90, 33, 60]
pass_students = 0
fail_students = 0

for i in marks:
    if i >= 50:
        pass_students = pass_students + 1
    else:
        fail_students = fail_students + 1

print("Total Pass:", pass_students)
print("Total Fail:", fail_students)

print("-------------------")

# 3. Simple Data Cleaner
names = [" Alice", "bob", " CHARLIE "]
clean_names = []

for n in names:
    # removing space and making lower case
    val = n.strip()
    val = val.lower()
    clean_names.append(val)

print("Cleaned data:", clean_names)

print("-------------------")

# 4. Message Length Analyzer
messages = ["Hi", "Welcome to the platform", "OK"]

for msg in messages:
    print("length is", len(msg), "for", msg)
    if len(msg) > 10:
        print("Flag: message is too long")

print("-------------------")

# 5. Error Message Detector
logs = ["INFO", "ERROR", "WARNING", "ERROR"]
error_count = 0

# checking for errors in the list
for log in logs:
    if log == "ERROR":
        error_count += 1

print("Total errors:", error_count)

Login Successful
-------------------
Total Pass: 3
Total Fail: 2
-------------------
Cleaned data: ['alice', 'bob', 'charlie']
-------------------
length is 2 for Hi
length is 23 for Welcome to the platform
Flag: message is too long
length is 2 for OK
-------------------
Total errors: 2
